# 🏭 파쇄기 앙상블 통합 건강점수 분석

**목적:** 9개 AI 모델의 출력을 통합하여 파쇄기의 종합 건강점수를 산출합니다.

| 구분 | 내용 |
|------|------|
| **데이터 규모** | 7일 × 5분 간격 = 2,016 샘플 (운영 수준) |
| **모델 수** | 9개 (베어링, 블레이드, 불균형, 재밍, 화재, 분진, 가스, 파쇄크기, RPM) |
| **도메인** | 유지보수(40%), 안전(35%), 품질(25%) |
| **점수 범위** | 0~100 (높을수록 건강) |
| **경보 수준** | Excellent(>85), Good(70-85), Fair(50-70), Poor(<50) |

---

## Step 0. 라이브러리 설치 및 임포트

In [ ]:
!pip install -q scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Matplotlib configuration
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('All libraries loaded successfully.')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')

---
## Step 1. 9개 모델 출력 시뮬레이션 (7일, 2,016 샘플)

각 모델의 출력을 현실적으로 시뮬레이션합니다. **인과관계**를 반영합니다:
- 베어링 열화 → 진동 증가 → 파쇄 크기 품질 저하
- 화재 이벤트 → 가스 농도 증가 → 안전 경보
- 재밍 발생 → 전류 스파이크 → 마모 가속

각 모델 출력은 **이상 확률 (0~1)** 형태입니다. 값이 높을수록 이상 가능성이 높습니다.

In [ ]:
np.random.seed(42)

# --- Time index: 7 days, 5-min intervals ---
N = 2016  # 7 days * 24 hours * 12 (5-min intervals)
start_time = datetime(2026, 3, 24, 0, 0, 0)
timestamps = [start_time + timedelta(minutes=5 * i) for i in range(N)]
t = np.arange(N)

# --- Helper functions ---
def daily_pattern(t, amplitude=0.05, phase=0):
    """Simulate daily operational cycle."""
    return amplitude * np.sin(2 * np.pi * t / 288 + phase)

def weekly_pattern(t, amplitude=0.03):
    """Simulate weekly pattern."""
    return amplitude * np.sin(2 * np.pi * t / 2016)

def trend(t, slope=0.00005):
    """Gradual degradation trend."""
    return slope * t

def noise(n, scale=0.02):
    """Random noise."""
    return np.random.normal(0, scale, n)

# =============================================================
# Model 1: Bearing Degradation
# =============================================================
bearing_base = 0.15 + trend(t, 0.00008) + daily_pattern(t, 0.04, 0)
bearing_event = np.zeros(N)
bearing_event[1100:1250] = np.linspace(0, 0.35, 150)
bearing_event[1250:1350] = np.linspace(0.35, 0.15, 100)
bearing_event[1700:1800] = np.linspace(0, 0.25, 100)
bearing_event[1800:1870] = np.linspace(0.25, 0.10, 70)
bearing = np.clip(bearing_base + bearing_event + noise(N, 0.02), 0, 1)

# =============================================================
# Model 2: Blade Wear
# =============================================================
blade_wear = np.clip(
    0.10 + trend(t, 0.00012) + daily_pattern(t, 0.03, 1.0)
    + weekly_pattern(t, 0.02) + noise(N, 0.015), 0, 1)

# =============================================================
# Model 3: Imbalance (causal: bearing -> vibration)
# =============================================================
imbalance = np.clip(
    0.12 + 0.3 * bearing + daily_pattern(t, 0.03, 0.5)
    + noise(N, 0.025), 0, 1)

# =============================================================
# Model 4: Jamming Detection
# =============================================================
jamming_base = 0.08 + daily_pattern(t, 0.03, 2.0) + noise(N, 0.02)
jam_events = np.zeros(N)
for jam_start in [400, 850, 1400, 1750]:
    duration = np.random.randint(20, 50)
    jam_events[jam_start:jam_start+duration] = np.random.uniform(0.4, 0.7, min(duration, N-jam_start))
    decay_len = min(80, N - jam_start - duration)
    if decay_len > 0:
        jam_events[jam_start+duration:jam_start+duration+decay_len] = \
            np.linspace(jam_events[jam_start+duration-1] if jam_start+duration-1 < N else 0.4, 0.05, decay_len)
jamming = np.clip(jamming_base + jam_events, 0, 1)

# Causal: Jamming -> blade wear acceleration
blade_wear = np.clip(blade_wear + 0.15 * jamming, 0, 1)

# =============================================================
# Model 5: Fire Risk Detection
# =============================================================
fire_base = 0.05 + daily_pattern(t, 0.02, 3.0) + noise(N, 0.015)
fire_event = np.zeros(N)
fire_event[1400:1430] = np.linspace(0, 0.6, 30)
fire_event[1430:1460] = np.linspace(0.6, 0.7, 30)
fire_event[1460:1530] = np.linspace(0.7, 0.1, 70)
fire = np.clip(fire_base + fire_event, 0, 1)

# =============================================================
# Model 6: Dust Concentration
# =============================================================
dust = np.clip(
    0.20 + daily_pattern(t, 0.06, 1.5) + trend(t, 0.00003)
    + 0.1 * jamming + noise(N, 0.025), 0, 1)

# =============================================================
# Model 7: Gas Leak (causal: fire -> gas)
# =============================================================
gas_leak = np.clip(
    0.08 + 0.4 * fire + daily_pattern(t, 0.02, 2.5)
    + noise(N, 0.02), 0, 1)

# =============================================================
# Model 8: Crush Size Quality (causal: bearing + blade -> quality)
# =============================================================
crush_size = np.clip(
    0.12 + 0.25 * bearing + 0.20 * blade_wear
    + daily_pattern(t, 0.03, 0.8) + noise(N, 0.02), 0, 1)

# =============================================================
# Model 9: RPM Quality
# =============================================================
rpm_quality = np.clip(
    0.10 + 0.2 * imbalance + 0.15 * jamming
    + daily_pattern(t, 0.02, 1.2) + noise(N, 0.018), 0, 1)

# =============================================================
# Build DataFrame
# =============================================================
df = pd.DataFrame({
    'timestamp': timestamps,
    'M1_bearing': bearing,
    'M2_blade_wear': blade_wear,
    'M3_imbalance': imbalance,
    'M4_jamming': jamming,
    'M5_fire': fire,
    'M6_dust': dust,
    'M7_gas_leak': gas_leak,
    'M8_crush_size': crush_size,
    'M9_rpm_quality': rpm_quality,
})
df.set_index('timestamp', inplace=True)

model_cols = [c for c in df.columns if c.startswith('M')]

print(f'\ub370\uc774\ud130 \uc0dd\uc131 \uc644\ub8cc: {df.shape[0]} \uc0d8\ud50c, {df.shape[1]} \ubaa8\ub378')
print(f'\uae30\uac04: {df.index[0]} ~ {df.index[-1]}')
print(f'\n--- \uae30\uc220 \ud1b5\uacc4 ---')
df[model_cols].describe().round(4)

---
## Step 2. 데이터 시각화

9개 모델 출력의 시계열 개요와 모델 간 상관관계를 확인합니다.

In [ ]:
# --- 2-1. All 9 model outputs overview ---
fig, axes = plt.subplots(9, 1, figsize=(16, 20), sharex=True)

colors = ['#e74c3c', '#e67e22', '#f39c12', '#2ecc71', '#e74c3c',
          '#9b59b6', '#3498db', '#1abc9c', '#34495e']
labels = ['Bearing Degradation', 'Blade Wear', 'Imbalance',
          'Jamming', 'Fire Risk', 'Dust Concentration',
          'Gas Leak', 'Crush Size Anomaly', 'RPM Quality Anomaly']

for i, (col, color, label) in enumerate(zip(model_cols, colors, labels)):
    axes[i].plot(df.index, df[col], color=color, linewidth=0.6, alpha=0.8)
    axes[i].fill_between(df.index, df[col], alpha=0.15, color=color)
    axes[i].set_ylabel(label, fontsize=8, fontweight='bold')
    axes[i].set_ylim(-0.05, 1.05)
    axes[i].axhline(y=0.5, color='gray', linestyle='--', alpha=0.4)
    axes[i].tick_params(labelsize=7)

axes[0].set_title('All 9 Model Outputs - 7 Day Overview (Anomaly Probability)',
                   fontsize=14, fontweight='bold', pad=10)
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d %H:%M'))
axes[-1].xaxis.set_major_locator(mdates.DayLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# --- 2-2. Correlation between model outputs ---
corr = df[model_cols].corr()

short_labels = ['Bearing', 'Blade', 'Imbalance', 'Jamming', 'Fire',
                'Dust', 'Gas', 'CrushSize', 'RPM']

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdYlBu_r', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(9))
ax.set_yticks(range(9))
ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short_labels, fontsize=9)

for i in range(9):
    for j in range(9):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=8, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, shrink=0.8, label='Correlation')
ax.set_title('Model Output Correlation Matrix (9 Models)',
             fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

print('\uc8fc\uc694 \uc0c1\uad00\uad00\uacc4:')
for i in range(9):
    for j in range(i+1, 9):
        if abs(corr.values[i, j]) > 0.5:
            print(f'  {short_labels[i]} <-> {short_labels[j]}: {corr.values[i,j]:.3f}')

---
## Step 3. 도메인별 점수 산출

이상 확률을 **건강 점수(0~100)**로 변환합니다. `건강점수 = (1 - 이상확률) × 100`

| 도메인 | 가중치 | 구성 모델 (가중치) |
|--------|--------|-------------------|
| **유지보수** | 40% | 베어링(30%) + 블레이드(30%) + 불균형(20%) + 재밍(20%) |
| **안전** | 35% | 화재(40%) + 분진(30%) + 가스(30%) |
| **품질** | 25% | 파쇄크기(50%) + RPM(50%) |

In [ ]:
# --- Convert anomaly probability to health score (0~100) ---
# Health = (1 - anomaly_prob) * 100

# Domain 1: Maintenance (40%)
df['maintenance_score'] = (
    0.30 * (1 - df['M1_bearing']) +
    0.30 * (1 - df['M2_blade_wear']) +
    0.20 * (1 - df['M3_imbalance']) +
    0.20 * (1 - df['M4_jamming'])
) * 100

# Domain 2: Safety (35%)
df['safety_score'] = (
    0.40 * (1 - df['M5_fire']) +
    0.30 * (1 - df['M6_dust']) +
    0.30 * (1 - df['M7_gas_leak'])
) * 100

# Domain 3: Quality (25%)
df['quality_score'] = (
    0.50 * (1 - df['M8_crush_size']) +
    0.50 * (1 - df['M9_rpm_quality'])
) * 100

domain_cols = ['maintenance_score', 'safety_score', 'quality_score']

print('--- \ub3c4\uba54\uc778\ubcc4 \uc810\uc218 \ud1b5\uacc4 ---')
print(df[domain_cols].describe().round(2))
print(f'\n\uc720\uc9c0\ubcf4\uc218 \ud3c9\uade0: {df["maintenance_score"].mean():.1f}')
print(f'\uc548\uc804 \ud3c9\uade0:     {df["safety_score"].mean():.1f}')
print(f'\ud488\uc9c8 \ud3c9\uade0:     {df["quality_score"].mean():.1f}')

---
## Step 4. 통합 건강점수 산출

**통합 건강점수** = 유지보수 × 0.40 + 안전 × 0.35 + 품질 × 0.25

| 등급 | 점수 범위 | 의미 |
|------|-----------|------|
| **Excellent** | > 85 | 정상 운전 |
| **Good** | 70 ~ 85 | 양호, 모니터링 |
| **Fair** | 50 ~ 70 | 주의, 점검 필요 |
| **Poor** | < 50 | 위험, 즉시 조치 |

In [ ]:
# --- Integrated Health Score ---
DOMAIN_WEIGHTS = {'maintenance': 0.40, 'safety': 0.35, 'quality': 0.25}

df['health_score'] = (
    DOMAIN_WEIGHTS['maintenance'] * df['maintenance_score'] +
    DOMAIN_WEIGHTS['safety']      * df['safety_score'] +
    DOMAIN_WEIGHTS['quality']     * df['quality_score']
)

# --- Alert Level Classification ---
def classify_alert(score):
    if score > 85:
        return 'Excellent'
    elif score > 70:
        return 'Good'
    elif score > 50:
        return 'Fair'
    else:
        return 'Poor'

df['alert_level'] = df['health_score'].apply(classify_alert)

# --- Summary Statistics ---
print('=' * 55)
print('      \ud1b5\ud569 \uac74\uac15\uc810\uc218 (Integrated Health Score)')
print('=' * 55)
print(f'  \ud3c9\uade0:   {df["health_score"].mean():.2f}')
print(f'  \uc911\uc559\uac12: {df["health_score"].median():.2f}')
print(f'  \ucd5c\uc18c:   {df["health_score"].min():.2f}')
print(f'  \ucd5c\ub300:   {df["health_score"].max():.2f}')
print(f'  \ud45c\uc900\ud3b8\ucc28: {df["health_score"].std():.2f}')
print('=' * 55)
print('\n--- \uacbd\ubcf4 \uc218\uc900 \ubd84\ud3ec ---')
alert_counts = df['alert_level'].value_counts()
for level in ['Excellent', 'Good', 'Fair', 'Poor']:
    count = alert_counts.get(level, 0)
    pct = count / len(df) * 100
    print(f'  {level:10s}: {count:5d} ({pct:5.1f}%)')

---
## Step 5. 교차 도메인 상관분석

- 9개 모델 간 상관행렬
- Random Forest 피처 중요도
- 인과 체인 식별

In [ ]:
# --- 5-1. Random Forest Feature Importance ---
X = df[model_cols].values
y = df['health_score'].values

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X, y)

importances = rf.feature_importances_
importance_df = pd.DataFrame({
    'model': short_labels,
    'importance': importances
}).sort_values('importance', ascending=False)

print('--- Random Forest \ud53c\ucc98 \uc911\uc694\ub3c4 (\uac74\uac15\uc810\uc218 \uc608\uce21) ---')
for _, row in importance_df.iterrows():
    bar = '#' * int(row['importance'] * 100)
    print(f'  {row["model"]:12s}: {row["importance"]:.4f}  {bar}')

rf_r2 = rf.score(X, y)
print(f'\nR-squared: {rf_r2:.4f}')

# --- 5-2. Causal chain identification ---
print('\n--- \uc778\uacfc \uccb4\uc778 \uc2dd\ubcc4 ---')
print('  Chain 1: Bearing Degradation -> Imbalance -> Crush Size Quality Drop')
print(f'    Bearing-Imbalance corr: {corr.loc["M1_bearing","M3_imbalance"]:.3f}')
print(f'    Bearing-CrushSize corr: {corr.loc["M1_bearing","M8_crush_size"]:.3f}')
print('  Chain 2: Fire Event -> Gas Leak Increase')
print(f'    Fire-Gas corr: {corr.loc["M5_fire","M7_gas_leak"]:.3f}')
print('  Chain 3: Jamming -> Blade Wear Acceleration -> Dust Increase')
print(f'    Jamming-Blade corr: {corr.loc["M4_jamming","M2_blade_wear"]:.3f}')
print(f'    Jamming-Dust corr:  {corr.loc["M4_jamming","M6_dust"]:.3f}')

---
## Step 6. 시각화

각 차트를 개별 셀에서 생성합니다.

### 6-1. 통합 건강점수 타임라인 (7일) + 도메인별 분해

In [ ]:
# --- 6-1. Integrated Health Score Timeline with Domain Breakdown ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(df.index, df['health_score'], color='#2c3e50', linewidth=1.0,
         label='Integrated Health Score', zorder=3)

ax1.axhspan(85, 100, alpha=0.15, color='green', label='Excellent (>85)')
ax1.axhspan(70, 85, alpha=0.15, color='#f1c40f', label='Good (70-85)')
ax1.axhspan(50, 70, alpha=0.15, color='orange', label='Fair (50-70)')
ax1.axhspan(0, 50, alpha=0.15, color='red', label='Poor (<50)')

rolling_avg = df['health_score'].rolling(window=36, center=True).mean()
ax1.plot(df.index, rolling_avg, color='#e74c3c', linewidth=2.0,
         linestyle='--', label='3-Hour Rolling Average', zorder=4)

ax1.set_ylabel('Health Score (0-100)', fontsize=11, fontweight='bold')
ax1.set_ylim(30, 100)
ax1.legend(loc='lower left', fontsize=8, ncol=3)
ax1.set_title('Integrated Health Score - 7 Day Timeline',
              fontsize=14, fontweight='bold', pad=10)

ax2.plot(df.index, df['maintenance_score'], color='#3498db', linewidth=0.8,
         label='Maintenance (40%)', alpha=0.8)
ax2.plot(df.index, df['safety_score'], color='#e74c3c', linewidth=0.8,
         label='Safety (35%)', alpha=0.8)
ax2.plot(df.index, df['quality_score'], color='#2ecc71', linewidth=0.8,
         label='Quality (25%)', alpha=0.8)

ax2.set_ylabel('Domain Score (0-100)', fontsize=11, fontweight='bold')
ax2.set_ylim(30, 100)
ax2.legend(loc='lower left', fontsize=9)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax2.xaxis.set_major_locator(mdates.DayLocator())
ax2.set_xlabel('Date', fontsize=11)

plt.tight_layout()
plt.show()

### 6-2. 모델 1~9 출력 히트맵 (정규화)

In [ ]:
# --- 6-2. Model Output Heatmap (Normalized) ---
df_hourly = df[model_cols].resample('1h').mean()

scaler = MinMaxScaler()
heatmap_data = scaler.fit_transform(df_hourly.values)

fig, ax = plt.subplots(figsize=(18, 6))
im = ax.imshow(heatmap_data.T, aspect='auto', cmap='YlOrRd',
               interpolation='nearest', vmin=0, vmax=1)

ax.set_yticks(range(9))
ax.set_yticklabels(short_labels, fontsize=10)

hour_ticks = list(range(0, len(df_hourly), 24))
hour_labels = [df_hourly.index[i].strftime('%m/%d') for i in hour_ticks if i < len(df_hourly)]
ax.set_xticks(hour_ticks[:len(hour_labels)])
ax.set_xticklabels(hour_labels, fontsize=10)
ax.set_xlabel('Date', fontsize=11)

cbar = plt.colorbar(im, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label('Normalized Anomaly Level', fontsize=10)

ax.set_title('Model Output Heatmap - 7 Day Overview (Hourly Avg, Normalized)',
             fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

### 6-3. 교차 도메인 상관행렬

In [ ]:
# --- 6-3. Cross-Domain Correlation Matrix ---
all_cols = model_cols + domain_cols + ['health_score']
all_labels = short_labels + ['Maint.Score', 'Safety Score', 'Quality Score', 'Health Score']

corr_full = df[all_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr_full.values, cmap='RdYlBu_r', vmin=-1, vmax=1, aspect='auto')

ax.set_xticks(range(len(all_labels)))
ax.set_yticks(range(len(all_labels)))
ax.set_xticklabels(all_labels, rotation=55, ha='right', fontsize=8)
ax.set_yticklabels(all_labels, fontsize=8)

for i in range(len(all_labels)):
    for j in range(len(all_labels)):
        val = corr_full.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=6, color=color)

for pos in [4, 7, 9, 12]:
    if pos < len(all_labels):
        ax.axhline(y=pos-0.5, color='black', linewidth=1.5)
        ax.axvline(x=pos-0.5, color='black', linewidth=1.5)

plt.colorbar(im, ax=ax, shrink=0.7, label='Correlation')
ax.set_title('Cross-Domain Correlation Matrix (Models + Domain Scores + Health)',
             fontsize=12, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()

### 6-4. 피처 중요도 (Random Forest)

In [ ]:
# --- 6-4. Feature Importance for Health Score (RF) ---
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(12, 6))

sorted_imp = importance_df.sort_values('importance', ascending=True)
domain_map = {
    'Bearing': '#3498db', 'Blade': '#3498db', 'Imbalance': '#3498db', 'Jamming': '#3498db',
    'Fire': '#e74c3c', 'Dust': '#e74c3c', 'Gas': '#e74c3c',
    'CrushSize': '#2ecc71', 'RPM': '#2ecc71'
}
colors_imp = [domain_map[m] for m in sorted_imp['model']]

bars = ax.barh(sorted_imp['model'], sorted_imp['importance'],
               color=colors_imp, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, sorted_imp['importance']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

legend_elements = [
    Patch(facecolor='#3498db', label='Maintenance Domain'),
    Patch(facecolor='#e74c3c', label='Safety Domain'),
    Patch(facecolor='#2ecc71', label='Quality Domain'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

ax.set_xlabel('Feature Importance', fontsize=11, fontweight='bold')
ax.set_title(f'Random Forest Feature Importance for Health Score (R\u00b2 = {rf_r2:.4f})',
             fontsize=13, fontweight='bold', pad=10)
ax.set_xlim(0, sorted_imp['importance'].max() * 1.15)
plt.tight_layout()
plt.show()

### 6-5. 도메인 점수 비교 (레이더 차트 + 막대 차트)

In [ ]:
# --- 6-5. Domain Score Comparison (Radar + Bar) ---
fig = plt.figure(figsize=(18, 7))

# --- Left: Radar chart ---
ax1 = fig.add_subplot(131, polar=True)
categories = ['Maintenance', 'Safety', 'Quality']
avg_scores = [df['maintenance_score'].mean(), df['safety_score'].mean(),
              df['quality_score'].mean()]
min_scores = [df['maintenance_score'].min(), df['safety_score'].min(),
              df['quality_score'].min()]

N_cat = len(categories)
angles = [n / float(N_cat) * 2 * np.pi for n in range(N_cat)]
angles += angles[:1]
avg_vals = avg_scores + [avg_scores[0]]
min_vals = min_scores + [min_scores[0]]

ax1.plot(angles, avg_vals, 'o-', linewidth=2, color='#2ecc71', label='Average')
ax1.fill(angles, avg_vals, alpha=0.15, color='#2ecc71')
ax1.plot(angles, min_vals, 'o-', linewidth=2, color='#e74c3c', label='Minimum')
ax1.fill(angles, min_vals, alpha=0.15, color='#e74c3c')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories, fontsize=10, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.set_title('Domain Scores\n(Avg vs Min)', fontsize=12, fontweight='bold', pad=20)
ax1.legend(loc='lower right', fontsize=8)

# --- Center: Grouped bar chart ---
ax2 = fig.add_subplot(132)
df_daily = df[domain_cols].resample('1D').mean()
days = [d.strftime('%m/%d') for d in df_daily.index]
x = np.arange(len(days))
width = 0.25

ax2.bar(x - width, df_daily['maintenance_score'], width, label='Maintenance',
        color='#3498db', alpha=0.8)
ax2.bar(x, df_daily['safety_score'], width, label='Safety',
        color='#e74c3c', alpha=0.8)
ax2.bar(x + width, df_daily['quality_score'], width, label='Quality',
        color='#2ecc71', alpha=0.8)
ax2.set_xticks(x)
ax2.set_xticklabels(days, fontsize=9)
ax2.set_ylabel('Score (0-100)', fontsize=10)
ax2.set_xlabel('Date', fontsize=10)
ax2.set_ylim(50, 100)
ax2.legend(fontsize=8)
ax2.set_title('Daily Average Domain Scores', fontsize=12, fontweight='bold')

# --- Right: Stacked contribution ---
ax3 = fig.add_subplot(133)
maint_contrib = df_daily['maintenance_score'] * 0.40
safety_contrib = df_daily['safety_score'] * 0.35
quality_contrib = df_daily['quality_score'] * 0.25

ax3.bar(x, maint_contrib, width=0.6, label='Maintenance (40%)', color='#3498db')
ax3.bar(x, safety_contrib, width=0.6, bottom=maint_contrib,
        label='Safety (35%)', color='#e74c3c')
ax3.bar(x, quality_contrib, width=0.6, bottom=maint_contrib + safety_contrib,
        label='Quality (25%)', color='#2ecc71')
ax3.set_xticks(x)
ax3.set_xticklabels(days, fontsize=9)
ax3.set_ylabel('Weighted Score Contribution', fontsize=10)
ax3.set_xlabel('Date', fontsize=10)
ax3.legend(fontsize=8)
ax3.set_title('Health Score Composition by Domain', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### 6-6. 경보 수준 타임라인 및 분포

In [ ]:
# --- 6-6. Alert Level Timeline and Distribution ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6),
                                gridspec_kw={'width_ratios': [3, 1]})

alert_colors = {
    'Excellent': '#27ae60', 'Good': '#f1c40f',
    'Fair': '#e67e22', 'Poor': '#e74c3c'
}
alert_order = ['Excellent', 'Good', 'Fair', 'Poor']

for level in alert_order:
    mask = df['alert_level'] == level
    if mask.sum() > 0:
        ax1.scatter(df.index[mask], df['health_score'][mask],
                   c=alert_colors[level], label=level, s=3, alpha=0.5)

for threshold, label in [(85, 'Excellent'), (70, 'Good'), (50, 'Fair')]:
    ax1.axhline(y=threshold, color='gray', linestyle=':', alpha=0.5)
    ax1.text(df.index[-1], threshold + 1, f'{threshold}', fontsize=8,
             color='gray', ha='right')

ax1.set_ylabel('Health Score', fontsize=11, fontweight='bold')
ax1.set_xlabel('Date', fontsize=11)
ax1.set_title('Alert Level Timeline - 7 Days', fontsize=13, fontweight='bold')
ax1.legend(markerscale=3, fontsize=9, loc='lower left')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax1.xaxis.set_major_locator(mdates.DayLocator())
ax1.set_ylim(30, 100)

counts = [df['alert_level'].value_counts().get(level, 0) for level in alert_order]
colors_pie = [alert_colors[level] for level in alert_order]
non_zero = [(level, cnt, col) for level, cnt, col in zip(alert_order, counts, colors_pie) if cnt > 0]

if non_zero:
    labels_nz, counts_nz, colors_nz = zip(*non_zero)
    wedges, texts, autotexts = ax2.pie(
        counts_nz, labels=labels_nz, colors=colors_nz,
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 9}
    )
    for autotext in autotexts:
        autotext.set_fontweight('bold')

ax2.set_title('Alert Level Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print('--- \uc77c\ubcc4 \uacbd\ubcf4 \uc218\uc900 \uc694\uc57d ---')
df['date'] = df.index.date
daily_alert = df.groupby('date')['alert_level'].value_counts().unstack(fill_value=0)
for level in alert_order:
    if level not in daily_alert.columns:
        daily_alert[level] = 0
daily_alert = daily_alert[alert_order]
print(daily_alert.to_string())
df.drop(columns=['date'], inplace=True)

---
## Step 7. 종합 요약

### 분석 결과

In [ ]:
# --- Step 7. Summary Report ---
print('=' * 65)
print('       \ud30c\uc1c4\uae30 \uc559\uc0c1\ube14 \ud1b5\ud569 \uac74\uac15\uc810\uc218 \ubd84\uc11d - \ucd5c\uc885 \uc694\uc57d')
print('=' * 65)

print(f'\n[\ub370\uc774\ud130 \uaddc\ubaa8]')
print(f'  \uae30\uac04:          {df.index[0].strftime("%Y-%m-%d")} ~ {df.index[-1].strftime("%Y-%m-%d")} (7\uc77c)')
print(f'  \uc0d8\ud50c \uc218:       {len(df):,} (5\ubd84 \uac04\uaca9)')
print(f'  \ubaa8\ub378 \uc218:       9\uac1c')

print(f'\n[\ud1b5\ud569 \uac74\uac15\uc810\uc218]')
print(f'  \ud3c9\uade0 \uc810\uc218:     {df["health_score"].mean():.1f} / 100')
print(f'  \ucd5c\uc800 \uc810\uc218:     {df["health_score"].min():.1f} (\uc2dc\uc810: {df["health_score"].idxmin().strftime("%m/%d %H:%M")})')
print(f'  \ucd5c\uace0 \uc810\uc218:     {df["health_score"].max():.1f}')

print(f'\n[\ub3c4\uba54\uc778\ubcc4 \ud3c9\uade0 \uc810\uc218]')
print(f'  \uc720\uc9c0\ubcf4\uc218 (40%): {df["maintenance_score"].mean():.1f}')
print(f'  \uc548\uc804     (35%): {df["safety_score"].mean():.1f}')
print(f'  \ud488\uc9c8     (25%): {df["quality_score"].mean():.1f}')

print(f'\n[\uacbd\ubcf4 \uc218\uc900 \ubd84\ud3ec]')
alert_counts = df['alert_level'].value_counts()
for level in ['Excellent', 'Good', 'Fair', 'Poor']:
    count = alert_counts.get(level, 0)
    pct = count / len(df) * 100
    print(f'  {level:10s}: {count:5d}\uac74 ({pct:5.1f}%)')

print(f'\n[\uc8fc\uc694 \uc778\uacfc \uccb4\uc778]')
print(f'  1. \ubca0\uc5b4\ub9c1 \uc5f4\ud654 \u2192 \ubd88\uade0\ud615 \uc99d\uac00 \u2192 \ud30c\uc1c4 \ud06c\uae30 \ud488\uc9c8 \uc800\ud558')
print(f'  2. \ud654\uc7ac \uc704\ud5d8 \u2192 \uac00\uc2a4 \ub204\ucd9c \uc99d\uac00 \u2192 \uc548\uc804 \uc810\uc218 \ud558\ub77d')
print(f'  3. \uc7ac\ubc0d \ubc1c\uc0dd \u2192 \ube14\ub808\uc774\ub4dc \ub9c8\ubaa8 \uac00\uc18d \u2192 \ubd84\uc9c4 \uc99d\uac00')

print(f'\n[Random Forest \ubd84\uc11d]')
print(f'  \uac74\uac15\uc810\uc218 \uc608\uce21 R\u00b2: {rf_r2:.4f}')
top3 = importance_df.head(3)
print(f'  \uc0c1\uc704 3 \uc911\uc694 \ud53c\ucc98:')
for _, row in top3.iterrows():
    print(f'    - {row["model"]}: {row["importance"]:.4f}')

print(f'\n[\uad8c\uc7a5 \uc870\uce58]')
worst_domain = min(
    [('\uc720\uc9c0\ubcf4\uc218', df['maintenance_score'].mean()),
     ('\uc548\uc804', df['safety_score'].mean()),
     ('\ud488\uc9c8', df['quality_score'].mean())],
    key=lambda x: x[1]
)
print(f'  \uac00\uc7a5 \ucde8\uc57d\ud55c \ub3c4\uba54\uc778: {worst_domain[0]} ({worst_domain[1]:.1f}\uc810)')
print(f'  \u2192 \ud574\ub2f9 \ub3c4\uba54\uc778\uc758 \uad6c\uc131 \ubaa8\ub378\uc5d0 \ub300\ud55c \uc9d1\uc911 \uc810\uac80 \ud544\uc694')
print(f'  \u2192 \uc778\uacfc \uccb4\uc778\uc744 \uace0\ub824\ud55c \uc608\ubc29 \uc815\ube44 \uacc4\ud68d \uc218\ub9bd \uad8c\uc7a5')
print('=' * 65)